# 🧪 TriFeaturizer - Multi-domain Molecular Feature Extractor

Turn a **protein/peptide sequence**, a **small-molecule SMILES**, or an **inorganic material** (chemical formula or CIF crystal structure) into a complete, machine-learning-ready feature set — either one item at a time, or for a whole uploaded dataset at once.

| Part | Input | Engines (run together) | Features produced |
|---|---|---|---|
| **1 · Sequence** | residue / peptide / protein | Biopython + peptides.py | ≈ 108 |
| **2 · Molecule** | SMILES | RDKit + mordredcommunity | 1613 |
| **3 · Material** | chemical formula **and/or** CIF | pymatgen + matminer + DScribe | 142 (+ SOAP for CIF) |

Within each part the engines work **together**: one produces the readable, explained values; the other emits the full descriptor vector.

## Two ways to use it

**A · Explore one item.** Paste a single sequence, SMILES, or formula and click **Analyze** for the explained table, **Charts** for quick plots, or **Full vector → CSV** for the complete descriptor set of that one item.

**B · Process a whole dataset → Excel.** Upload a **CSV, Excel, or JSON** file containing a column of sequences (or SMILES, or formulas). TriFeaturizer computes the *complete feature vector for every row* and returns a single **`.xlsx` workbook** — one row per item, one column per descriptor — ready to load straight into a model. The relevant column is auto-detected by name (`smiles`, `sequence`, `formula`, …); you can also type the column name explicitly.

## How to run

1. Run the **Shared utilities** cell once (top of the notebook).
2. For each part you need, run its **`pip install`** cell, then its **code** cell.
3. Use the single-item controls, or the **Upload list → full-feature Excel** button for batch mode.

> Each part installs only its own libraries, so a sequence-only user never pulls the materials stack.

> **Feature counts (full vector):** Sequence ≈ **108** (102 peptides.py + 6 Biopython) · Molecule **1613** (Mordred 2-D) · Composition **142** (132 Magpie + 4 valence-orbital + 6 stoichiometry) · CIF structure **8** + an optional **SOAP** block (≈ 390–1500, growing with the number of distinct elements). Full names and meanings are in the companion **Feature Reference** Word document.

> **If pip prints a red `requests` / `google-colab` version conflict:** it is a *harmless warning*, not an error — features still extract correctly. To silence it: `!pip -q install "requests==2.32.4"`, then **Runtime ▸ Restart session**.

In [1]:
# @title
# === Shared utilities — run this first ========================================
# Lightweight: only pandas + ipywidgets + matplotlib, all preinstalled in Colab.
import io
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.dpi": 120,     # crisp preview in the notebook
    "savefig.dpi": 600,    # exported PNGs are 600 dpi
    "savefig.bbox": "tight",
    "font.size": 11,
})

try:
    from google.colab import output as _co
    _co.enable_custom_widget_manager()
    IN_COLAB = True
except Exception:
    IN_COLAB = False


def render_feature_table(rows, caption=""):
    """rows = list of (feature, value, explanation)."""
    df = pd.DataFrame(rows, columns=["Feature", "Value", "What it means"])
    styler = (df.style.hide(axis="index")
              .set_table_styles([
                  {"selector": "th", "props": [("text-align", "left"),
                      ("background-color", "#1f2937"), ("color", "white"), ("padding", "6px 10px")]},
                  {"selector": "td", "props": [("text-align", "left"),
                      ("padding", "6px 10px"), ("border-bottom", "1px solid #e5e7eb")]}]))
    if caption:
        display(HTML(f"<h4 style='margin:8px 0'>{caption}</h4>"))
    display(styler)


def download_file(filename):
    """Offer a file for download in Colab; otherwise point to the Files panel."""
    if IN_COLAB:
        try:
            from google.colab import files
            files.download(filename)
            return
        except Exception:
            pass
    print(f"  Saved to {filename} (open the Files panel on the left to download).")


def download_dataframe(df, filename):
    df.to_csv(filename, index=False)
    print(f"\u2713 Wrote {df.shape[0]} row(s) \u00d7 {df.shape[1]} column(s) \u2192 {filename}")
    download_file(filename)


def write_excel(df, filename, failed=None):
    """Write a full feature matrix to .xlsx and offer it for download."""
    df.to_excel(filename, index=False)
    print(f"\u2713 Feature matrix: {df.shape[0]} row(s) \u00d7 {df.shape[1]} column(s) \u2192 {filename}")
    if failed:
        preview = ", ".join(map(str, failed[:3]))
        print(f"\u26a0 Skipped {len(failed)} invalid entr(y/ies): {preview}{' \u2026' if len(failed) > 3 else ''}")
    download_file(filename)


def read_uploaded_text():
    """Upload a single text file (FASTA / CIF) and return (text, filename)."""
    if not IN_COLAB:
        print("Upload works inside Colab \u2014 paste the text instead."); return None, None
    from google.colab import files
    up = files.upload()
    if not up:
        return None, None
    name = next(iter(up))
    return up[name].decode("utf-8", errors="ignore"), name


def read_uploaded_table():
    """Upload a CSV / Excel / JSON dataset and return (DataFrame, filename)."""
    if not IN_COLAB:
        print("Upload works inside Colab. (Locally, build a DataFrame yourself.)"); return None, None
    from google.colab import files
    up = files.upload()
    if not up:
        return None, None
    name = next(iter(up)); raw = up[name]; low = name.lower()
    try:
        if low.endswith((".xlsx", ".xls")):
            df = pd.read_excel(io.BytesIO(raw))
        elif low.endswith(".json"):
            df = pd.read_json(io.BytesIO(raw))
        else:
            df = pd.read_csv(io.BytesIO(raw))
    except Exception as e:
        print("Could not read that file:", e); return None, None
    print(f"Loaded {name}: {df.shape[0]} rows \u00d7 {df.shape[1]} columns \u2192 {list(df.columns)}")
    return df, name


def pick_column(df, candidates, label="input"):
    """Find the input column by common names; fall back to the first column."""
    low = {str(c).strip().lower(): c for c in df.columns}
    for n in candidates:
        if n in low:
            return low[n]
    print(f"No obvious {label} column found \u2014 using the first column '{df.columns[0]}'.")
    return df.columns[0]


def example_button(label, value, target):
    b = widgets.Button(description=label, layout=widgets.Layout(width="auto"))
    b.on_click(lambda _: setattr(target, "value", value))
    return b


def save_and_show(fig, filename):
    """Save a 600-dpi PNG (and download it in Colab), then display the figure."""
    fig.savefig(filename, dpi=600, bbox_inches="tight")
    print(f"\u2713 Saved 600-dpi image \u2192 {filename}")
    download_file(filename)
    plt.show()


def batch_panel(default_help, on_run):
    """Standard upload-and-process panel: column box + button, shared across parts."""
    col = widgets.Text(value="", placeholder="column name (blank = auto-detect)",
                       layout=widgets.Layout(width="55%"))
    btn = widgets.Button(description="Upload list \u2192 full-feature Excel",
                         button_style="success", icon="table",
                         layout=widgets.Layout(width="auto"))
    btn.on_click(lambda _: on_run(col.value.strip()))
    return widgets.VBox([widgets.HTML(f"<i>{default_help}</i>"),
                         widgets.HBox([col, btn])])


print("Shared utilities ready. Running in Colab:", IN_COLAB)

Shared utilities ready. Running in Colab: True


## Part 1 · Sequence features  (amino acid · peptide · protein)

**Biopython (`ProtParam`)** supplies the interpretable physico-chemical properties — molecular weight, isoelectric point, net charge, GRAVY hydropathy, aromaticity, instability index, and extinction coefficient. **peptides.py** adds 102 published QSAR descriptors (Kidera, Z-scales, VHSE, BLOSUM indices, and more) that make up the bulk of the full vector. A single residue falls back to a curated reference table.

*Batch mode:* upload a file with a `sequence` column to get the full ≈108-feature matrix for every sequence as an Excel workbook.

In [2]:
!pip -q install biopython peptides openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 4.0 MB/s eta 0:00:00


In [3]:
# @title
# === PART 1 \u00b7 Sequence features ===========================================
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import peptides

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")
SEQ_COLS = ["sequence", "seq", "peptide", "protein", "fasta", "sequences"]

# (name, 3-letter, MW, pI, Kyte-Doolittle hydropathy, side-chain class)
AA_TABLE = {
 'A':('Alanine','Ala',89.09,6.00,1.8,'nonpolar'), 'R':('Arginine','Arg',174.20,10.76,-4.5,'positive'),
 'N':('Asparagine','Asn',132.12,5.41,-3.5,'polar'), 'D':('Aspartate','Asp',133.10,2.77,-3.5,'negative'),
 'C':('Cysteine','Cys',121.16,5.07,2.5,'polar'),  'Q':('Glutamine','Gln',146.15,5.65,-3.5,'polar'),
 'E':('Glutamate','Glu',147.13,3.22,-3.5,'negative'),'G':('Glycine','Gly',75.07,5.97,-0.4,'nonpolar'),
 'H':('Histidine','His',155.16,7.59,-3.2,'positive'),'I':('Isoleucine','Ile',131.17,6.02,4.5,'nonpolar'),
 'L':('Leucine','Leu',131.17,5.98,3.8,'nonpolar'), 'K':('Lysine','Lys',146.19,9.74,-3.9,'positive'),
 'M':('Methionine','Met',149.21,5.74,1.9,'nonpolar'),'F':('Phenylalanine','Phe',165.19,5.48,2.8,'nonpolar'),
 'P':('Proline','Pro',115.13,6.30,-1.6,'nonpolar'),'S':('Serine','Ser',105.09,5.68,-0.8,'polar'),
 'T':('Threonine','Thr',119.12,5.60,-0.7,'polar'), 'W':('Tryptophan','Trp',204.23,5.89,-0.9,'nonpolar'),
 'Y':('Tyrosine','Tyr',181.19,5.66,-1.3,'polar'),  'V':('Valine','Val',117.15,5.96,4.2,'nonpolar')}


def clean_sequence(text):
    body = "".join(l for l in str(text).strip().splitlines() if not l.strip().startswith(">"))
    return "".join(c for c in body.upper() if c.isalpha())


def featurize_sequence_full(text):
    """Return the complete sequence feature dict (102 peptides.py + 6 Biopython), or None."""
    clean = "".join(c for c in clean_sequence(text) if c in STANDARD_AA)
    if not clean:
        return None
    d = peptides.Peptide(clean).descriptors()
    pa = ProteinAnalysis(clean)
    d.update({"Length": len(clean), "MW": pa.molecular_weight(),
              "pI": pa.isoelectric_point(), "GRAVY": pa.gravy(),
              "Instability": pa.instability_index(), "Aromaticity": pa.aromaticity()})
    return d


def analyze_sequence(text):
    seq = clean_sequence(text)
    if not seq:
        print("Please paste a sequence (raw or FASTA)."); return
    if len(seq) == 1 and seq in AA_TABLE:
        name, tlc, mw, pI, kd, cls = AA_TABLE[seq]
        render_feature_table([
            ("Amino acid", f"{name} ({tlc}, {seq})", "Residue identity"),
            ("Molecular weight (Da)", mw, "Free amino-acid mass"),
            ("Isoelectric point (pI)", pI, "pH of zero net charge"),
            ("Hydropathy (Kyte\u2013Doolittle)", kd, "+ hydrophobic / \u2212 hydrophilic"),
            ("Side-chain class", cls, "Polarity / charge")],
            caption=f"Single residue \u00b7 {name}")
        return
    nonstd = sorted(set(seq) - STANDARD_AA)
    if nonstd:
        print("\u26a0 Ignoring non-standard characters:", nonstd)
    clean = "".join(c for c in seq if c in STANDARD_AA)
    if not clean:
        print("No standard amino acids found."); return
    pa = ProteinAnalysis(clean)
    ext = pa.molar_extinction_coefficient()
    rows = [
        ("Length (residues)", len(clean), "Chain length"),
        ("Molecular weight (Da)", round(pa.molecular_weight(), 2), "Peptide / protein mass"),
        ("Isoelectric point (pI)", round(pa.isoelectric_point(), 2), "pH of zero net charge"),
        ("Net charge @ pH 7.0", round(pa.charge_at_pH(7.0), 2), "Charge at neutral pH"),
        ("GRAVY", round(pa.gravy(), 3), "Avg hydropathy; + = hydrophobic"),
        ("Aromaticity", round(pa.aromaticity(), 3), "Fraction Phe+Trp+Tyr"),
        ("Instability index", round(pa.instability_index(), 2), "> 40 \u21d2 likely unstable in vitro"),
        ("Extinction coeff (reduced)", ext[0], "M\u207b\u00b9cm\u207b\u00b9 @280 nm, Cys reduced")]
    pep = peptides.Peptide(clean)
    try:
        hm = round(pep.hydrophobic_moment(window=min(11, len(clean))), 3)
    except Exception:
        hm = "n/a"
    rows += [
        ("Aliphatic index", round(pep.aliphatic_index(), 2), "Thermostability proxy (A,V,I,L)"),
        ("Boman index", round(pep.boman(), 2), "Protein-binding potential (kcal/mol)"),
        ("Hydrophobic moment", hm, "Amphipathicity")]
    render_feature_table(rows, caption=f"Sequence features \u00b7 {len(clean)} residues")


def full_sequence_vector(text):
    d = featurize_sequence_full(text)
    if d is None:
        print("Please paste a valid sequence first."); return
    download_dataframe(pd.DataFrame([d]), "part1_sequence_features.csv")


def batch_sequences(df, col):
    rows, failed = [], []
    for v in df[col].astype(str):
        v = v.strip()
        if not v or v.lower() == "nan":
            continue
        d = featurize_sequence_full(v)
        if d is None:
            failed.append(v); continue
        rows.append({"input_sequence": v, **d})
    if not rows:
        print("No valid sequences found in that column."); return
    write_excel(pd.DataFrame(rows), "part1_sequence_FULL_features.xlsx", failed)


def chart_sequence(text):
    from collections import Counter
    clean = "".join(c for c in clean_sequence(text) if c in STANDARD_AA)
    if len(clean) < 2:
        print("Need a peptide/protein (\u22652 residues) to draw charts."); return
    order = "ACDEFGHIKLMNPQRSTVWY"
    counts = Counter(clean)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
    ax[0].bar(list(order), [counts.get(a, 0) for a in order], color="#2563eb")
    ax[0].set_title("Amino-acid composition"); ax[0].set_ylabel("Count")
    kd = {a: AA_TABLE[a][4] for a in order}
    w = min(9, len(clean))
    prof = [sum(kd[clean[i + j]] for j in range(w)) / w for i in range(len(clean) - w + 1)]
    ax[1].plot(range(1, len(prof) + 1), prof, color="#dc2626")
    ax[1].axhline(0, color="gray", lw=0.8)
    ax[1].set_title(f"Kyte\u2013Doolittle hydropathy (window {w})")
    ax[1].set_xlabel("Residue position"); ax[1].set_ylabel("Hydropathy")
    fig.suptitle(f"Sequence charts \u00b7 {len(clean)} residues", fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    save_and_show(fig, "chart_sequence.png")

# ---- UI ----------------------------------------------------------------------
p1_in = widgets.Textarea(value="DAEFRHDSGYEVHHQK",
    placeholder="Paste a residue, peptide, or protein sequence (raw or FASTA)\u2026",
    layout=widgets.Layout(width="100%", height="90px"))
p1_out = widgets.Output()
p1_a = widgets.Button(description="Analyze", button_style="primary", icon="search")
p1_f = widgets.Button(description="Full vector \u2192 CSV", icon="download")
p1_c = widgets.Button(description="Charts", icon="bar-chart")

def _p1a(_):
    with p1_out: clear_output(); analyze_sequence(p1_in.value)
def _p1f(_):
    with p1_out: clear_output(); full_sequence_vector(p1_in.value)
def _p1c(_):
    with p1_out: clear_output(); chart_sequence(p1_in.value)
p1_a.on_click(_p1a); p1_f.on_click(_p1f); p1_c.on_click(_p1c)

def _p1_run_batch(colname):
    with p1_out:
        clear_output()
        df, name = read_uploaded_table()
        if df is None: return
        col = colname or pick_column(df, SEQ_COLS, "sequence")
        print(f"Featurizing column '{col}' \u2026")
        batch_sequences(df, col)

display(widgets.VBox([
    widgets.HTML("<b>Single sequence \u2014 examples:</b>"),
    widgets.HBox([example_button("\u03b2-amyloid frag", "DAEFRHDSGYEVHHQK", p1_in),
                  example_button("Insulin A-chain", "GIVEQCCTSICSLYQLENYCN", p1_in),
                  example_button("Single residue (W)", "W", p1_in)]),
    p1_in, widgets.HBox([p1_a, p1_f, p1_c]),
    widgets.HTML("<hr><b>Batch a dataset</b>"),
    batch_panel("Upload a CSV / Excel / JSON with a column of sequences "
                "(e.g. <code>sequence</code>). Returns one Excel row per sequence.", _p1_run_batch),
    p1_out]))

## Part 2 · Molecule features  (SMILES)

**RDKit** provides the interpretable, structure-aware descriptors (molecular weight, LogP, TPSA, hydrogen-bond counts, QED drug-likeness, Lipinski rule-of-five, Murcko scaffold) plus a 2-D depiction. **mordredcommunity** (the maintained Mordred fork) generates the full **1613-descriptor** 2-D vector.

*Batch mode:* upload a file with a `smiles` column to get the full 1613-feature matrix for every molecule as an Excel workbook.

In [4]:
!pip -q install rdkit mordredcommunity openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.0/176.0 kB 6.0 MB/s eta 0:00:00


In [5]:
# @title
# === PART 2 \u00b7 Small-molecule features (SMILES) ============================
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, Crippen, Lipinski, QED, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from mordred import Calculator, descriptors as _mordred_desc

SMILES_COLS = ["smiles", "smile", "structure", "canonical_smiles", "smiles_string"]
_MORDRED = Calculator(_mordred_desc, ignore_3D=True)   # 1613 2-D descriptors


def analyze_smiles(s):
    mol = Chem.MolFromSmiles(str(s).strip())
    if mol is None:
        print("Could not parse that SMILES \u2014 check the string."); return
    mw, logp = Descriptors.MolWt(mol), Crippen.MolLogP(mol)
    hbd, hba = Lipinski.NumHDonors(mol), Lipinski.NumHAcceptors(mol)
    viol = sum([mw > 500, logp > 5, hbd > 5, hba > 10])
    render_feature_table([
        ("Canonical SMILES", Chem.MolToSmiles(mol), "RDKit-normalised structure"),
        ("Formula", rdMolDescriptors.CalcMolFormula(mol), "Molecular formula"),
        ("Molecular weight", round(mw, 2), "g/mol"),
        ("LogP (Crippen)", round(logp, 2), "Lipophilicity"),
        ("TPSA", round(rdMolDescriptors.CalcTPSA(mol), 2), "Polar surface area (\u00c5\u00b2)"),
        ("H-bond donors", hbd, "Lipinski rule: \u2264 5"),
        ("H-bond acceptors", hba, "Lipinski rule: \u2264 10"),
        ("Rotatable bonds", Descriptors.NumRotatableBonds(mol), "Flexibility"),
        ("Aromatic rings", rdMolDescriptors.CalcNumAromaticRings(mol), "Ring count"),
        ("Heavy atoms", mol.GetNumHeavyAtoms(), "Non-H atoms"),
        ("QED", round(QED.qed(mol), 3), "Drug-likeness, 0\u20131"),
        ("Lipinski violations", viol, "Ro5: 0\u20131 typical for drugs"),
        ("Murcko scaffold", MurckoScaffold.MurckoScaffoldSmiles(mol=mol), "Core ring system")],
        caption="Molecular descriptors")
    display(Draw.MolToImage(mol, size=(360, 260)))


def full_smiles_vector(s):
    mol = Chem.MolFromSmiles(str(s).strip())
    if mol is None:
        print("Could not parse that SMILES."); return
    df = _MORDRED.pandas([mol]).apply(pd.to_numeric, errors="coerce")
    df.insert(0, "SMILES", Chem.MolToSmiles(mol))
    download_dataframe(df, "part2_molecule_features.csv")


def batch_smiles(df, col):
    keep, mols, failed = [], [], []
    for s in df[col].astype(str):
        s = s.strip()
        if not s or s.lower() == "nan":
            continue
        m = Chem.MolFromSmiles(s)
        if m is None:
            failed.append(s)
        else:
            mols.append(m); keep.append(Chem.MolToSmiles(m))
    if not mols:
        print("No valid SMILES found in that column."); return
    feat = _MORDRED.pandas(mols).apply(pd.to_numeric, errors="coerce")
    feat.insert(0, "input_SMILES", keep)
    write_excel(feat, "part2_molecule_FULL_features.xlsx", failed)


def chart_smiles(s):
    import numpy as np
    mol = Chem.MolFromSmiles(str(s).strip())
    if mol is None:
        print("Parse a valid SMILES first."); return
    mw, logp = Descriptors.MolWt(mol), Crippen.MolLogP(mol)
    hbd, hba = Lipinski.NumHDonors(mol), Lipinski.NumHAcceptors(mol)
    tpsa = rdMolDescriptors.CalcTPSA(mol)
    labels = ["MW/500", "LogP/5", "HBD/5", "HBA/10", "TPSA/140"]
    ratios = [max(0, mw / 500), max(0, logp / 5), hbd / 5, hba / 10, tpsa / 140]
    ang = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist(); ang += ang[:1]
    vals = ratios + ratios[:1]
    fig = plt.figure(figsize=(6.6, 7.2))
    ax = plt.subplot(111, polar=True)
    ax.set_theta_offset(np.pi / 2); ax.set_theta_direction(-1)
    ax.plot(ang, vals, color="#2563eb", linewidth=2)
    ax.fill(ang, vals, alpha=0.25, color="#2563eb")
    ax.plot(ang, [1] * len(ang), "--", color="#dc2626", linewidth=1.5,
            label="Rule-of-5 limit  (inside the ring = passes)")
    ax.set_xticks(ang[:-1]); ax.set_xticklabels(labels, fontsize=11)
    ax.set_rlabel_position(36); ax.set_ylim(0, max(1.25, max(vals) * 1.1))
    fig.suptitle("Drug-likeness vs. rule-of-5", y=0.97, fontsize=14, fontweight="bold")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.07), frameon=False, fontsize=10)
    fig.subplots_adjust(top=0.84, bottom=0.15, left=0.10, right=0.90)
    save_and_show(fig, "chart_molecule_druglikeness.png")

# ---- UI ----
p2_in = widgets.Textarea(value="CC(=O)Oc1ccccc1C(=O)O",
    placeholder="Paste a SMILES string\u2026",
    layout=widgets.Layout(width="100%", height="70px"))
p2_out = widgets.Output()
p2_a = widgets.Button(description="Analyze", button_style="primary", icon="search")
p2_f = widgets.Button(description="Full vector \u2192 CSV", icon="download")
p2_c = widgets.Button(description="Charts", icon="bar-chart")

def _p2a(_):
    with p2_out: clear_output(); analyze_smiles(p2_in.value)
def _p2f(_):
    with p2_out: clear_output(); full_smiles_vector(p2_in.value)
def _p2c(_):
    with p2_out: clear_output(); chart_smiles(p2_in.value)
p2_a.on_click(_p2a); p2_f.on_click(_p2f); p2_c.on_click(_p2c)

def _p2_run_batch(colname):
    with p2_out:
        clear_output()
        df, name = read_uploaded_table()
        if df is None: return
        col = colname or pick_column(df, SMILES_COLS, "SMILES")
        print(f"Featurizing column '{col}' with Mordred (1613 descriptors) \u2026")
        batch_smiles(df, col)

display(widgets.VBox([
    widgets.HTML("<b>Single molecule \u2014 examples:</b>"),
    widgets.HBox([example_button("Aspirin", "CC(=O)Oc1ccccc1C(=O)O", p2_in),
                  example_button("Caffeine", "CN1C=NC2=C1C(=O)N(C(=O)N2C)C", p2_in),
                  example_button("Ibuprofen", "CC(C)Cc1ccc(C(C)C(=O)O)cc1", p2_in)]),
    p2_in, widgets.HBox([p2_a, p2_f, p2_c]),
    widgets.HTML("<hr><b>Batch a dataset</b>"),
    batch_panel("Upload a CSV / Excel / JSON with a column of SMILES "
                "(e.g. <code>smiles</code>). Returns one Excel row per molecule \u00d7 1613 descriptors.",
                _p2_run_batch),
    p2_out]))

## Part 3 · Material features  (chemical formula and/or CIF structure)

Two complementary sub-tools that share one install:

- **3a · Composition** — from a formula alone (e.g. `LiFePO4`). Uses **pymatgen** + **matminer** Magpie element-property statistics, valence-orbital fractions, and stoichiometry norms (142 features).
- **3b · Structure (CIF)** — from a crystal file. Uses **pymatgen** to parse and **matminer** `DensityFeatures` / `GlobalSymmetryFeatures`, plus an optional **DScribe SOAP** fingerprint.

> ⚠ matminer is built for **inorganic** compositions and crystals. Rule of thumb: *organic / connectivity matters → Part 2 (SMILES); oxide, alloy, crystal → Part 3.* This is the heaviest install (pymatgen + ASE) — allow a couple of minutes.

*Batch mode (composition):* upload a file with a `formula` column to get the full 142-feature matrix for every compound as an Excel workbook.

In [6]:
# @title
!pip -q install matminer dscribe openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 777.4/777.4 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.1/829.1 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━

In [7]:
# @title
# === PART 3 \u00b7 Material features (composition + CIF structure) =============
import numpy as np
from pymatgen.core import Composition, Structure, Lattice
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from matminer.featurizers.composition import ElementProperty, ValenceOrbital, Stoichiometry

_ep = ElementProperty.from_preset("magpie")
_vo = ValenceOrbital(props=["frac"])
_st = Stoichiometry()
FORMULA_COLS = ["formula", "composition", "compound", "formulae", "compounds", "material"]

CURATED_MAGPIE = [
    ("MagpieData mean Number", "Mean atomic number"),
    ("MagpieData mean AtomicWeight", "Mean atomic weight"),
    ("MagpieData mean Electronegativity", "Mean Pauling electronegativity"),
    ("MagpieData mean CovalentRadius", "Mean covalent radius (pm)"),
    ("MagpieData mean NValence", "Mean valence-electron count"),
    ("MagpieData avg_dev Electronegativity", "Spread of electronegativity")]


def featurize_composition_full(formula):
    try:
        comp = Composition(formula)
    except Exception:
        return None
    d = {}
    for f in (_ep, _vo, _st):
        d.update(dict(zip(f.feature_labels(), f.featurize(comp))))
    return d


# ---- 3a composition ----------------------------------------------------------
def analyze_composition(formula):
    try:
        comp = Composition(formula)
    except Exception as e:
        print("Could not parse formula:", e); return
    ep = dict(zip(_ep.feature_labels(), _ep.featurize(comp)))
    vo = dict(zip(_vo.feature_labels(), _vo.featurize(comp)))
    rows = [
        ("Reduced formula", comp.reduced_formula, "Normalised composition"),
        ("Elements", ", ".join(sorted(e.symbol for e in comp.elements)), "Constituents"),
        ("Atoms / formula unit", round(comp.num_atoms, 3), "Total atoms"),
        ("Avg atomic mass", round(float(comp.weight) / comp.num_atoms, 3), "amu per atom")]
    for lbl, meaning in CURATED_MAGPIE:
        if lbl in ep:
            rows.append((lbl.replace("MagpieData ", "Magpie "), round(ep[lbl], 3), meaning))
    for lbl in _vo.feature_labels():
        rows.append((lbl, round(vo[lbl], 3), "Fraction of valence e\u207b in this orbital"))
    render_feature_table(rows, caption=f"Composition features \u00b7 {comp.reduced_formula}")


def full_composition_vector(formula):
    d = featurize_composition_full(formula)
    if d is None:
        print("Could not parse that formula."); return
    df = pd.DataFrame([d]); df.insert(0, "formula", Composition(formula).reduced_formula)
    download_dataframe(df, "part3_composition_features.csv")


def batch_compositions(df, col):
    rows, failed = [], []
    for v in df[col].astype(str):
        v = v.strip()
        if not v or v.lower() == "nan":
            continue
        d = featurize_composition_full(v)
        if d is None:
            failed.append(v); continue
        rows.append({"input_formula": v, **d})
    if not rows:
        print("No valid formulas found in that column."); return
    write_excel(pd.DataFrame(rows), "part3_composition_FULL_features.xlsx", failed)


def chart_composition(formula):
    try:
        comp = Composition(formula)
    except Exception as e:
        print("Could not parse formula:", e); return
    fr = comp.fractional_composition.get_el_amt_dict()
    els = list(fr.keys())
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(els, [fr[e] for e in els], color="#0891b2")
    ax.set_title(f"Atomic fraction \u00b7 {comp.reduced_formula}")
    ax.set_ylabel("Fraction"); ax.set_ylim(0, 1)
    fig.tight_layout()
    save_and_show(fig, "chart_composition.png")


# ---- 3b structure / CIF ------------------------------------------------------
NACL = Structure.from_spacegroup("Fm-3m", Lattice.cubic(5.64),
                                 ["Na", "Cl"], [[0, 0, 0], [0.5, 0.5, 0.5]])


def analyze_structure(st):
    sga = SpacegroupAnalyzer(st)
    a, b, c = st.lattice.abc
    render_feature_table([
        ("Formula", st.composition.reduced_formula, "Structure composition"),
        ("Sites in cell", len(st), "Atoms in the unit cell"),
        ("Density (g/cm\u00b3)", round(float(st.density), 3), "Mass / volume"),
        ("Cell volume (\u00c5\u00b3)", round(st.volume, 2), "Unit-cell volume"),
        ("Volume / atom (\u00c5\u00b3)", round(st.volume / len(st), 2), "Packing measure"),
        ("Lattice a,b,c (\u00c5)", f"{a:.3f}, {b:.3f}, {c:.3f}", "Cell edge lengths"),
        ("Space group", f"{sga.get_space_group_symbol()} (#{sga.get_space_group_number()})", "Symmetry"),
        ("Crystal system", sga.get_crystal_system(), "Lattice family")],
        caption="Crystal-structure features")


def full_structure_vector(st, with_soap=False):
    from matminer.featurizers.structure import DensityFeatures, GlobalSymmetryFeatures
    data = {}
    for f in (DensityFeatures(), GlobalSymmetryFeatures()):
        try:
            data.update(dict(zip(f.feature_labels(), f.featurize(st))))
        except Exception as e:
            print("skipped", f.__class__.__name__, ":", e)
    if with_soap:
        from dscribe.descriptors import SOAP
        from pymatgen.io.ase import AseAtomsAdaptor
        atoms = AseAtomsAdaptor.get_atoms(st)
        species = sorted(set(atoms.get_chemical_symbols()))
        soap = SOAP(species=species, r_cut=5.0, n_max=6, l_max=4, periodic=True)
        vec = soap.create(atoms).mean(axis=0)
        data.update({f"SOAP_{i}": v for i, v in enumerate(vec)})
    df = pd.DataFrame([data]); df.insert(0, "formula", st.composition.reduced_formula)
    download_dataframe(df, "part3_structure_features.csv")


def chart_structure(st):
    a, b, c = st.lattice.abc
    fr = st.composition.fractional_composition.get_el_amt_dict()
    els = list(fr.keys())
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].bar(els, [fr[e] for e in els], color="#0891b2")
    ax[0].set_title(f"Atomic fraction \u00b7 {st.composition.reduced_formula}")
    ax[0].set_ylabel("Fraction"); ax[0].set_ylim(0, 1)
    ax[1].bar(["a", "b", "c"], [a, b, c], color="#7c3aed")
    ax[1].set_title("Lattice parameters (\u00c5)")
    fig.tight_layout()
    save_and_show(fig, "chart_structure.png")


def load_structure(cif_text):
    return Structure.from_str(cif_text, fmt="cif")

# ---- UI: 3a composition ----
p3c_in = widgets.Text(value="LiFePO4", placeholder="Chemical formula, e.g. Fe2O3",
                      layout=widgets.Layout(width="60%"))
p3c_out = widgets.Output()
p3c_a = widgets.Button(description="Analyze", button_style="primary", icon="search")
p3c_f = widgets.Button(description="Full vector \u2192 CSV", icon="download")
p3c_c = widgets.Button(description="Charts", icon="bar-chart")

def _p3ca(_):
    with p3c_out: clear_output(); analyze_composition(p3c_in.value)
def _p3cf(_):
    with p3c_out: clear_output(); full_composition_vector(p3c_in.value)
def _p3cc(_):
    with p3c_out: clear_output(); chart_composition(p3c_in.value)
p3c_a.on_click(_p3ca); p3c_f.on_click(_p3cf); p3c_c.on_click(_p3cc)

def _p3c_run_batch(colname):
    with p3c_out:
        clear_output()
        df, name = read_uploaded_table()
        if df is None: return
        col = colname or pick_column(df, FORMULA_COLS, "formula")
        print(f"Featurizing column '{col}' (142 composition descriptors) \u2026")
        batch_compositions(df, col)

display(widgets.VBox([
    widgets.HTML("<h4>3a \u00b7 Composition (formula only)</h4><b>Examples:</b>"),
    widgets.HBox([example_button("LiFePO\u2084", "LiFePO4", p3c_in),
                  example_button("Fe\u2082O\u2083", "Fe2O3", p3c_in),
                  example_button("BaTiO\u2083", "BaTiO3", p3c_in)]),
    p3c_in, widgets.HBox([p3c_a, p3c_f, p3c_c]),
    widgets.HTML("<hr><b>Batch a dataset</b>"),
    batch_panel("Upload a CSV / Excel / JSON with a column of formulas "
                "(e.g. <code>formula</code>). Returns one Excel row per compound \u00d7 142 descriptors.",
                _p3c_run_batch),
    p3c_out]))

# ---- UI: 3b structure / CIF ----
_state = {"struct": None}
p3s_in = widgets.Textarea(
    placeholder="Paste CIF text here, or click \u201cLoad NaCl example\u201d / \u201cUpload CIF\u201d\u2026",
    layout=widgets.Layout(width="100%", height="120px"))
p3s_out = widgets.Output()
p3s_soap = widgets.Checkbox(value=False, description="Include SOAP (DScribe) in CSV")
p3s_ex = widgets.Button(description="Load NaCl example", icon="cube")
p3s_up = widgets.Button(description="Upload CIF", icon="upload")
p3s_a = widgets.Button(description="Analyze structure", button_style="primary", icon="search")
p3s_f = widgets.Button(description="Full vector \u2192 CSV", icon="download")
p3s_c = widgets.Button(description="Charts", icon="bar-chart")

def _resolve():
    if p3s_in.value.strip():
        return load_structure(p3s_in.value)
    return _state["struct"]

def _p3sex(_):
    from pymatgen.io.cif import CifWriter
    _state["struct"] = NACL
    p3s_in.value = str(CifWriter(NACL))
def _p3sup(_):
    t, _n = read_uploaded_text()
    if t: p3s_in.value = t
def _p3sa(_):
    with p3s_out:
        clear_output(); st = _resolve()
        if st is None: print("Paste a CIF, upload one, or load the example."); return
        analyze_structure(st)
def _p3sf(_):
    with p3s_out:
        clear_output(); st = _resolve()
        if st is None: print("Paste a CIF, upload one, or load the example."); return
        full_structure_vector(st, with_soap=p3s_soap.value)
def _p3sc(_):
    with p3s_out:
        clear_output(); st = _resolve()
        if st is None: print("Paste a CIF, upload one, or load the example."); return
        chart_structure(st)
p3s_ex.on_click(_p3sex); p3s_up.on_click(_p3sup)
p3s_a.on_click(_p3sa); p3s_f.on_click(_p3sf); p3s_c.on_click(_p3sc)

display(widgets.VBox([
    widgets.HTML("<h4>3b \u00b7 Crystal structure (CIF)</h4>"),
    widgets.HBox([p3s_ex, p3s_up]),
    p3s_in, p3s_soap, widgets.HBox([p3s_a, p3s_f, p3s_c]), p3s_out]))